In [0]:


# IDENTITY COLUMNS - gold.data_quality (dq_log_id) and gold.market_price
# (market_price_id) use SQL-side IDENTITY surrogate keys, added in the DDL
# because their Databricks source data has no unique natural key (see the
# DDL's own comments: dq_log has none, and (ticker, bar_timestamp) in
# market_price is NOT unique - confirmed from a real run where 6 tickers
# map to more than one company). These two columns are deliberately NEVER
# written from Spark - the DataFrames below don't have them, and SQL Server
# auto-populates IDENTITY columns on insert as long as the insert statement
# doesn't try to supply them, which Spark's JDBC writer already does not
# (it only writes the columns present in the DataFrame schema).
#
# CREDENTIALS: pulled from a Databricks secret scope, never hardcoded -
# see the credential issue already found and fixed in
# 00_setup_silver_catalog.ipynb. Set up the scope once before running this:
#   databricks secrets create-scope pe-fund-secrets
#   databricks secrets put-secret pe-fund-secrets sql-jdbc-user
#   databricks secrets put-secret pe-fund-secrets sql-jdbc-password
# ============================================================================

In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F

## Config

Credentials are hardcoded below for a one-time run before presentation - see the reminder in the cell itself to blank them out before committing to GitHub.

In [0]:
AZURE_SQL_SERVER = "pe-platform-sql-server.database.windows.net"
AZURE_SQL_PORT = 1433
AZURE_SQL_DATABASE = "pe-platform-db"

# HARDCODED for a one-time run before presentation - blank these back to
# empty strings (or delete this cell's values) BEFORE committing to GitHub.
# See chat: rotate this password after presenting, since it has already
# been shared in plaintext once before.
SQL_USER = "Shreyash1"
SQL_PASSWORD = "Shreyash@123"

SQL_CONNECTION_PROPERTIES = {
    "host": AZURE_SQL_SERVER,
    "port": str(AZURE_SQL_PORT),
    "database": AZURE_SQL_DATABASE,
    "user": SQL_USER,
    "password": SQL_PASSWORD,
    "encrypt": "true",
    "trustServerCertificate": "false",
}

print(f"Target: {AZURE_SQL_SERVER}:{AZURE_SQL_PORT}/{AZURE_SQL_DATABASE}")

Target: pe-platform-sql-server.database.windows.net:1433/pe-platform-db


## Load helper

`truncate=true` keeps the DDL's table structure (PK, IDENTITY, indexes) intact across every run, instead of `overwrite` mode's default drop-and-recreate behavior (which would silently lose the primary keys and indexes DE-3 already set up from the DDL).

In [0]:
def load_gold_table_to_sql(table_name: str, sql_table: str, columns: list, df=None):
    """
    Reads dbw_pe_platform.gold.<table_name> (unless a DataFrame is passed
    directly), selects only the columns that exist in the Azure SQL DDL
    (dropping any extra lineage/tracking columns picked up along the way,
    e.g. _run_id/_ingested_at), and truncate-loads it into Azure SQL's
    gold.<sql_table>.
    """
    if df is None:
        df = spark.table(gold_table(table_name))

    df = df.select(*columns)
    row_count = df.count()

    (
        df.write
        .format("sqlserver")
        .option("host", SQL_CONNECTION_PROPERTIES["host"])
        .option("port", SQL_CONNECTION_PROPERTIES["port"])
        .option("database", SQL_CONNECTION_PROPERTIES["database"])
        .option("dbtable", f"gold.{sql_table}")
        .option("user", SQL_CONNECTION_PROPERTIES["user"])
        .option("password", SQL_CONNECTION_PROPERTIES["password"])
        .option("encrypt", SQL_CONNECTION_PROPERTIES["encrypt"])
        .option("trustServerCertificate", SQL_CONNECTION_PROPERTIES["trustServerCertificate"])
        .option("truncate", "true")
        .mode("overwrite")
        .save()
    )

    print(f"Loaded {row_count} rows: dbw_pe_platform.gold.{table_name} -> Azure SQL gold.{sql_table}")
    return row_count

## Load all 7 tables

Straightforward pass-throughs for the first six - column names already match the DDL exactly. `market_price` and `data_quality` need no special handling either, since their surrogate IDENTITY columns simply don't exist in the Databricks source DataFrames (see header note).

In [0]:
GOLD_TABLE_COLUMNS = {
    "fund_snapshot": [
        "fund_id", "fund_name", "vintage_year", "fund_size_usd",
        "total_commitments", "contributions", "uncalled_capital",
        "invested_capital", "portfolio_value", "available_cash_internal",
        "available_cash_external", "estimated_nav", "calculated_at",
        "snapshot_generated_at",
    ],
    "fund_financials": [
        "fund_id", "total_commitments", "contributions", "uncalled_capital",
        "invested_capital", "portfolio_value", "available_cash_internal",
        "available_cash_external", "estimated_nav", "calculated_at",
        "gold_loaded_at",
    ],
    "portfolio_valuation": [
        "company_id", "company_name", "fund_id", "benchmark_ticker",
        "latest_close", "quantity", "market_value", "has_benchmark",
        "has_position", "gold_loaded_at",
    ],
    "reconciliation": [
        "recon_id", "recon_type", "business_date", "fund_id", "entity_id",
        "source_a_value", "source_b_value", "source_a_numeric",
        "source_b_numeric", "difference", "status", "break_reason",
        "created_at", "gold_loaded_at",
    ],
    "payment_summary": [
        "fund_id", "payment_type", "source_side", "total_amount",
        "payment_count", "gold_loaded_at",
    ],
    "data_quality": [
        # dq_log_id is SQL-side IDENTITY - not written from Spark
        "source_name", "business_date", "rule_name", "records_checked",
        "records_failed", "reason_code", "failure_rate", "created_at",
        "gold_loaded_at",
    ],
    "market_price": [
        # market_price_id is SQL-side IDENTITY - not written from Spark
        "ticker", "bar_timestamp", "open", "high", "low", "close",
        "volume", "company_id", "company_name", "fund_id", "gold_loaded_at",
    ],
}

load_counts = {}

load_counts["fund_snapshot"] = load_gold_table_to_sql("fund_snapshot", "fund_snapshot", GOLD_TABLE_COLUMNS["fund_snapshot"])
load_counts["fund_financials"] = load_gold_table_to_sql("fund_financials", "fund_financials", GOLD_TABLE_COLUMNS["fund_financials"])
load_counts["portfolio_valuation"] = load_gold_table_to_sql("portfolio_valuation", "portfolio_valuation", GOLD_TABLE_COLUMNS["portfolio_valuation"])
load_counts["reconciliation"] = load_gold_table_to_sql("reconciliation", "reconciliation", GOLD_TABLE_COLUMNS["reconciliation"])
load_counts["payment_summary"] = load_gold_table_to_sql("payment_summary", "payment_summary", GOLD_TABLE_COLUMNS["payment_summary"])
load_counts["data_quality"] = load_gold_table_to_sql("data_quality", "data_quality", GOLD_TABLE_COLUMNS["data_quality"])
load_counts["market_price"] = load_gold_table_to_sql("market_price", "market_price", GOLD_TABLE_COLUMNS["market_price"])

Loaded 5 rows: dbw_pe_platform.gold.fund_snapshot -> Azure SQL gold.fund_snapshot
Loaded 5 rows: dbw_pe_platform.gold.fund_financials -> Azure SQL gold.fund_financials
Loaded 30 rows: dbw_pe_platform.gold.portfolio_valuation -> Azure SQL gold.portfolio_valuation
Loaded 55 rows: dbw_pe_platform.gold.reconciliation -> Azure SQL gold.reconciliation
Loaded 18 rows: dbw_pe_platform.gold.payment_summary -> Azure SQL gold.payment_summary
Loaded 53 rows: dbw_pe_platform.gold.data_quality -> Azure SQL gold.data_quality
Loaded 271 rows: dbw_pe_platform.gold.market_price -> Azure SQL gold.market_price


## Verify: row counts in Azure SQL match Databricks

Reads each table back from Azure SQL via JDBC and compares against the counts captured during load - confirms nothing was dropped or duplicated in transit.

In [0]:
print("Verification - Databricks count vs Azure SQL count:")
header = "{:<22}{:>12}{:>12}{:>8}".format("table", "databricks", "azure_sql", "match")
print(header)

all_match = True
for table_name, expected_count in load_counts.items():
    sql_df = (
        spark.read
        .format("sqlserver")
        .option("host", SQL_CONNECTION_PROPERTIES["host"])
        .option("port", SQL_CONNECTION_PROPERTIES["port"])
        .option("database", SQL_CONNECTION_PROPERTIES["database"])
        .option("dbtable", f"gold.{table_name}")
        .option("user", SQL_CONNECTION_PROPERTIES["user"])
        .option("password", SQL_CONNECTION_PROPERTIES["password"])
        .option("encrypt", SQL_CONNECTION_PROPERTIES["encrypt"])
        .option("trustServerCertificate", SQL_CONNECTION_PROPERTIES["trustServerCertificate"])
        .load()
    )
    actual_count = sql_df.count()
    is_match = actual_count == expected_count
    all_match = all_match and is_match
    row = "{:<22}{:>12}{:>12}{:>8}".format(table_name, expected_count, actual_count, str(is_match))
    print(row)

print()
if all_match:
    print("OK: every table\'s Azure SQL row count matches its Databricks source.")
else:
    print("WARNING: at least one table\'s row count does not match - check the mismatched table(s) above before treating this load as complete.")

Verification - Databricks count vs Azure SQL count:
table                   databricks   azure_sql   match
fund_snapshot                    5           5    True
fund_financials                  5           5    True
portfolio_valuation             30          30    True
reconciliation                  55          55    True
payment_summary                 18          18    True
data_quality                    53          53    True
market_price                   271         271    True

OK: every table's Azure SQL row count matches its Databricks source.


### Summary

In [0]:
print("=== 18_load_azure_sql complete ===")
total = sum(load_counts.values())
for t, c in load_counts.items():
    print(f"  gold.{t}: {c} rows")
print(f"  TOTAL: {total} rows loaded into Azure SQL")

=== 18_load_azure_sql complete ===
  gold.fund_snapshot: 5 rows
  gold.fund_financials: 5 rows
  gold.portfolio_valuation: 30 rows
  gold.reconciliation: 55 rows
  gold.payment_summary: 18 rows
  gold.data_quality: 53 rows
  gold.market_price: 271 rows
  TOTAL: 437 rows loaded into Azure SQL
